In [2]:
import os
import sys

from pymongo import MongoClient
from dotenv import load_dotenv
from fastapi import FastAPI

sys.path.append(os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', 'ingestion')))

from requests_utils import get_repo, filter_features
from preprocessing import preprocess_df

sys.path.append(os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', 'model')))

from Doc2VecModel import Doc2VecModel

sys.path.append(os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', 'database')))

from DatabaseManager import DatabaseManager

In [26]:
os.environ["MLFLOW_TRACKING_URI"] = "../../artifacts/mlruns"

production_mlflow_model_readme_uri = 'models:/doc2vec_readme@production'
production_mlflow_model_others_uri = 'models:/doc2vec_others@production'

client = MongoClient('localhost', 27017)
database_manager = DatabaseManager(client['github'])

features_for_json = [
    'full_name',
    'id',
    'description',
    'language',
    'topics',
    'contents_url',
    'html_url',
    'default_branch',
]

features_for_df = features_for_json + ['owner_id']

production_model = Doc2VecModel(
    'production',
    production_mlflow_model_readme_uri,
    production_mlflow_model_others_uri
)

env_path = '../../.env'
load_dotenv(env_path)
token = os.getenv("TOKEN_GITHUB")

df_vectors = database_manager.get_df_vectors()

In [27]:
def recommend(repo_name: str, k: int):
    # Requêter pour récupérer la réponse json
    print('get_repo')
    repo_json = get_repo(token=token, full_name=repo_name)
    
    print('filter_repo')
    # Prétraiter pour récupérer un dictionnaire avec readme_preproc et others_preproc
    repo_filtered = filter_features([repo_json], features_for_json)
    df = preprocess_df(
        token=token,
        repos=repo_filtered,
        features=features_for_df,
    )

    print('orienter dict')
    dict_repo = df.to_dict(orient='records')[0]

    print('chercher k similaires')
    # Récupération des k repos les plus similaires pour chaque représentation
    k_most_similar_with_readme, k_most_similar_with_others = (
        production_model.get_top_k_for_prediction(k, dict_repo, df_vectors)
    )

    # On considère seulement les ids
    print('chercher_urls')
    ids_with_readme = k_most_similar_with_readme['id'].tolist()
    ids_with_others = k_most_similar_with_others['id'].tolist()

    # Récupération des urls des repos similaires à partir des ids
    readme_urls = database_manager.get_url_list_from_id_list(ids_with_readme)
    others_urls = database_manager.get_url_list_from_id_list(ids_with_others)
    
    return {
        "recommandations en utilisant le readme": readme_urls,
        "recommandations en n'utilisant pas le readme": others_urls,
    }

In [10]:
name = 'ARBML/whisperar'

In [28]:
res = recommend(name, k=2)
print(res)

get_repo
Requêtes restantes (global): 4984
filter_repo
Requêtes restantes (global): 4983


/home/choux/Documents/M2/Projet_mlops_ingestion/GitMatch/src/ingestion/preprocessing.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['contents_url'] = df2.apply(lambda x: load_readme(x['contents_url'][:-7] + f'README.md?ref={x["default_branch"]}', token), axis=1)


orienter dict
chercher k similaires
chercher_urls
{'recommandations en utilisant le readme': ['https://github.com/BogiHsu/Tacotron2-PyTorch', 'https://github.com/TideDancer/interspeech21_emotion'], "recommandations en n'utilisant pas le readme": ['https://github.com/Datalux/Osintgram', 'https://github.com/BishopFox/GitGot']}


In [29]:
res

{'recommandations en utilisant le readme': ['https://github.com/BogiHsu/Tacotron2-PyTorch',
  'https://github.com/TideDancer/interspeech21_emotion'],
 "recommandations en n'utilisant pas le readme": ['https://github.com/Datalux/Osintgram',
  'https://github.com/BishopFox/GitGot']}